**Historical engineering notebook (Phase 8 Single-Agent smoke).** Not the official 420-case evaluation. Official evaluation: Phase 15 (benchmark) and Phase 16 (judge).


# V2 Phase 8 — Colab GPU Single-Agent RAG smoke

**Before running:** Runtime → Change runtime type → **GPU** (T4 or better).

This notebook:
1. Clones V2 from GitHub
2. Installs dependencies
3. **Downloads FinQA source PDFs and rebuilds the Chroma index** (Option B — no Mac DB copy)
4. Runs index preflight validation
5. Runs Phase 8 Single-Agent RAG smoke (`llama_cpp`, n=3)

## Setup instructions

Push latest V2 changes to branch `main`, then run all cells.

The knowledge base is **not** in GitHub. Cell 3 downloads 230 FinQA page PDFs from Hugging Face and builds Chroma (~10–20 min on T4).

**Outputs:** `results/config/phase8_smoke_test.json`, `phase8_single_agent_smoke.json`

## 1. Clone GitHub repo and enter V2

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'main'
CLONE_DIR = Path('/content/capstone-rag')

if CLONE_DIR.exists():
    !rm -rf {CLONE_DIR}

print('Cloning branch:', BRANCH)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'git clone failed. Push V2/ to GitHub on branch {BRANCH!r} first.')

V2_ROOT = CLONE_DIR / 'V2'
if not V2_ROOT.is_dir():
    raise FileNotFoundError(f'Missing V2/ folder at {V2_ROOT}')
if not (V2_ROOT / 'scripts' / 'build_index.py').is_file():
    raise FileNotFoundError(f'Invalid V2 root: {V2_ROOT}')

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
print('OK — working in V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Download FinQA PDFs and rebuild Chroma index (Option B)

Downloads source PDFs from `G4KMU/t2-ragbench` and builds a fresh index. Does **not** use the Mac Chroma database.

In [ ]:
!PYTHONPATH=. python scripts/build_index.py --distractors 50

## 4. Index preflight validation

In [ ]:
!PYTHONPATH=. python scripts/validate_kb_index.py

## 5. Phase 8 Single-Agent RAG smoke (llama_cpp, n=3)

In [ ]:
!PYTHONPATH=. python scripts/smoke_single_agent.py --backend llama_cpp --limit 3

## 6. Check results

In [ ]:
import json
from pathlib import Path

fp = Path('results/config/phase8_runtime_fingerprint.json')
smoke = Path('results/config/phase8_smoke_test.json')
detail = Path('results/config/phase8_single_agent_smoke.json')
print('fingerprint:', fp.is_file())
print('smoke_test:', smoke.is_file())
print('detail:', detail.is_file())
if smoke.is_file():
    data = json.loads(smoke.read_text())
    print('status:', data.get('status'))
    print('actual:', data.get('actual'))
if detail.is_file():
    detail_data = json.loads(detail.read_text())
    for case in detail_data.get('cases', []):
        print(
            case.get('question_id'),
            'n_evidence=',
            len(case.get('retrieved_evidence') or []),
            'answer_len=',
            len(case.get('answer') or ''),
        )

## 7. Save knowledge base + results to Google Drive

Persists the Chroma index, PDFs, and smoke JSONs so later Colab sessions (Phases 9–10) can restore from Drive instead of rebuilding.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

# Mount Google Drive (approve the popup once)
drive.mount('/content/drive')

V2 = Path('/content/capstone-rag/V2')  # or Path('.') if already in V2
DRIVE_ROOT = Path('/content/drive/MyDrive/MSc-RAG')

KB_INDEX_SRC = V2 / 'knowledge_base' / 'index'
KB_DOCS_SRC = V2 / 'knowledge_base' / 'documents'
KB_INDEX_DST = DRIVE_ROOT / 'artifacts' / 'knowledge_base' / 'index'
KB_DOCS_DST = DRIVE_ROOT / 'artifacts' / 'knowledge_base' / 'documents'
RESULTS_DST = DRIVE_ROOT / 'configs' / 'phase8'


def copy_tree(src: Path, dst: Path) -> None:
    if not src.is_dir():
        raise FileNotFoundError(f'Missing source folder: {src}')
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    n_files = sum(1 for _ in dst.rglob('*') if _.is_file())
    print(f'copied {src} -> {dst} ({n_files} files)')


# 1) Knowledge base (reuse across Phases 8–10 without rebuilding)
copy_tree(KB_INDEX_SRC, KB_INDEX_DST)
copy_tree(KB_DOCS_SRC, KB_DOCS_DST)

# 2) Smoke / manifest JSONs
RESULTS_DST.mkdir(parents=True, exist_ok=True)
for name in (
    'phase8_runtime_fingerprint.json',
    'phase8_smoke_test.json',
    'phase8_single_agent_smoke.json',
    'phase6_index_manifest.json',
):
    src = V2 / 'results' / 'config' / name
    if src.is_file():
        shutil.copy2(src, RESULTS_DST / name)
        print('copied', name)

print('\nDone. Persisted on Drive:')
print('  Index :', KB_INDEX_DST)
print('  PDFs  :', KB_DOCS_DST)
print('  JSONs :', RESULTS_DST)